In [ ]:
%%capture
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 --index-url https://download.pytorch.org/whl/cu121
!pip install transformers accelerate peft trl bitsandbytes
!pip install unsloth groq langchain langchain-huggingface faiss-cpu langchain-community

In [ ]:
from unsloth import FastVisionModel, get_chat_template
import torch
import json
import random
import time
import glob
import base64
import os
import zipfile
import shutil
import gc
from groq import Groq
from google.colab import drive, userdata
from IPython.display import display, Image
import PIL.Image

In [ ]:
# Configuration for Groq API (used for question generation and judging)
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

In [ ]:
# Mount Drive and define dataset path
drive.mount('/content/drive')
drive_path = '/content/drive/MyDrive/Image_Dataset.zip'

In [ ]:
target_dir = "/content/images"
if os.path.exists(drive_path):
    with zipfile.ZipFile(drive_path, 'r') as zip_ref:
        zip_ref.extractall(target_dir)
    print(f"Images extracted to {target_dir}")

In [ ]:
# Unzip the zip file containing the model
shutil.unpack_archive("pharma_visiontext_model.zip", "pharma_model", "zip")

In [ ]:
def encode_image(image_path):
    """
    Encodes images to Base64 for API calls.
    """
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')


def _call_groq(messages, model, temperature=0.7, max_tokens=1500, json_mode=False):
    payload = {"model": model, "messages": messages, "temperature": temperature, "max_tokens": max_tokens}
    if json_mode:
        payload["response_format"] = {"type": "json_object"}

    for attempt in range(3):
        try:
            resp = client.chat.completions.create(**payload)
            return resp.choices[0].message.content
        except Exception as e:
            print(f"Groq error ({attempt+1}/3): {e}")
            time.sleep(2 ** attempt)
    return None

In [ ]:
def question_generator(n, question_type, image_paths):
  """
  Generates pharmacological questions using an LLM.
  """
    result = []
    topics = ["mechanism of action", "side effects", "contraindications",
              "drug interactions", "therapeutic indications", "pharmacokinetics"]

    print(f"Generating {n} '{question_type}' questions")

    for i in range(n):
        img_path = None
        if question_type == "text":
            topic = random.choice(topics)
            instruction = f"Generate ONE unique pharmacological question about {topic}. Respond ONLY in JSON: {{\"question\": \"...\", \"ground_truth_hint\": \"...\"}}"
            messages = [{"role": "user", "content": instruction}]

        elif question_type == "vision":
            img_path = random.choice(image_paths)
            b64_img = encode_image(img_path)
            instruction = "Identify the medication in this image. Ask ONLY for its name. Respond ONLY in JSON: {\"question\": \"...\", \"ground_truth_hint\": \"...\"}"
            messages = [{"role": "user", "content": [
                {"type": "text", "text": instruction},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}}
            ]}]

        else: # vision_text
            img_path = random.choice(image_paths)
            b64_img = encode_image(img_path)

            instruction = ("Look at this medication. DO NOT ask for its name. "
                           "Instead, ask about its therapeutic use, dosage form, or a common side effect "
                           "based on the active ingredient you see. Respond ONLY in JSON: "
                           "{\"question\": \"...\", \"ground_truth_hint\": \"...\"}")
            messages = [{"role": "user", "content": [
                {"type": "text", "text": instruction},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64_img}"}}
            ]}]

        content = _call_groq(messages, model="meta-llama/llama-4-scout-17b-16e-instruct", json_mode=True)
        if content:
            try:
                data = json.loads(content)
                result.append({
                    "question": data.get("question"),
                    "type": question_type,
                    "image_path": img_path,
                    "ground_truth": data.get("ground_truth_hint")
                })
            except: pass
        time.sleep(1)
    return result

In [ ]:
def evaluate_with_judge(question, ground_truth, anonymous_responses, image_path=None):
    """
    Evaluates AI responses using llama-4-scout as a pharmacist judge.
    Metrics: Accuracy, Safety and Conciseness.
    """
    responses_str = "\n".join([f"=== RESPONSE {rid} ===\n{resp}" for rid, resp in anonymous_responses.items()])

    instruction = f"""
    You are an expert medical pharmacist. Evaluate the following responses.

    QUESTION: {question}
    GROUND TRUTH (Correct Answer): {ground_truth}

    RESPONSES TO EVALUATE:
    {responses_str}

    SCORING CRITERIA (1-10):
    1. Accuracy: How well does it match the Ground Truth?
    2. Safety: Is the medical information safe and non-hallucinated?
    3. Conciseness: Is it direct and clear?

    Return ONLY a JSON object exactly like this:
    {{
      "evaluations": {{
        "A": {{"accuracy": int, "safety": int, "conciseness": int}},
        "B": {{"accuracy": int, "safety": int, "conciseness": int}}
      }}
    }}
    """

    content_payload = [{"type": "text", "text": instruction}]
    if image_path:
        content_payload.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encode_image(image_path)}"}})

    content = _call_groq([{"role": "user", "content": content_payload}], model="meta-llama/llama-4-scout-17b-16e-instruct", json_mode=True)

    try:
        data = json.loads(content)
        return data.get('evaluations', data)
    except Exception as e:
        print(f" Judge Parsing Error: {e}")
        return {"A": {"accuracy": 1, "safety": 1, "conciseness": 1}, "B": {"accuracy": 1, "safety": 1, "conciseness": 1}}

In [ ]:
def build_evaluation_dataset(n_text, n_vision, n_vt, image_folder):
    print("Phase 1: Building the exam")
    image_paths = glob.glob(f"{image_folder}/**/*.*", recursive=True)
    dataset = []

    for q_type, count in [("text", n_text), ("vision", n_vision), ("vision_text", n_vt)]:
        if count > 0:
            questions = question_generator(count, q_type, image_paths)
            dataset.extend(questions)

    print(f"\nExam created with {len(dataset)} total questions.")
    return dataset

In [ ]:
IMAGE_FOLDER = "/content/images/Image_Dataset"
examen_dataset = build_evaluation_dataset(n_text=2, n_vision=2, n_vt=2, image_folder=IMAGE_FOLDER)

In [ ]:
def model_response(model, tokenizer, question, image_path=None):
    """
    Universal inference for the loaded model on GPU
    """
    content_payload = []

    if image_path and os.path.exists(image_path):
        img_obj = PIL.Image.open(image_path).convert("RGB")
        content_payload.extend([{"type": "image"}, {"type": "text", "text": f"Question: {question}"}])
    else:
        content_payload.append({"type": "text", "text": f"Question: {question}"})

    prompt = tokenizer.apply_chat_template([{"role": "user", "content": content_payload}], add_generation_prompt=True, tokenize=False)

    if image_path and os.path.exists(image_path):
        inputs = tokenizer(text=prompt, images=img_obj, return_tensors="pt", padding=True).to("cuda")
    else:
        inputs = tokenizer(text=prompt, return_tensors="pt", padding=True).to("cuda")

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.1, repetition_penalty=1.05, use_cache=True)

    answer = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0].split("model\n")[-1].strip()

    # Freeing VRAM
    del inputs, outputs
    torch.cuda.empty_cache()
    return answer

In [ ]:
print("Phase 2: Loading base model")

base_model, base_tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
    dtype = torch.float16,
)

base_tokenizer = get_chat_template(base_tokenizer, chat_template="gemma-3")
FastVisionModel.for_inference(base_model)

print("\nBase model answering the exam:")
for i, item in enumerate(examen_dataset, 1):
    print(f"   - Processing question {i}/{len(examen_dataset)} ({item['type']})")
    torch.cuda.empty_cache()
    item["respuesta_base"] = model_response(base_model, base_tokenizer, item["question"], item["image_path"])

# Path where the progress will be saved in your Drive
checkpoint_path = '/content/drive/MyDrive/partial_exam.json'

# Save the dataset including the base model responses
with open(checkpoint_path, 'w') as f:
    json.dump(examen_dataset, f)

print(f"Progress saved to {checkpoint_path}.")

# Freeing VRAM
del base_model, base_tokenizer
gc.collect()
torch.cuda.empty_cache()

vram_libre = torch.cuda.mem_get_info()[0] / 1e9
print(f"VRAM Freed. Available: {vram_libre:.2f} GB")

In [ ]:
checkpoint_path = '/content/drive/MyDrive/partial_exam.json'

with open(checkpoint_path, 'r') as f:
    examen_dataset = json.load(f)
    print(f"✅ Success: Exam loaded with {len(examen_dataset)} questions.")
    print("Now you can proceed to load the Fine-Tuned model and generate its responses.")

In [ ]:
print("Phase 3: Loading Fine-tuned model")

my_model, my_tokenizer = FastVisionModel.from_pretrained(
    model_name = "pharma_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)

In [ ]:
FastVisionModel.for_inference(my_model)

print("\nFine-tuned model answering the exam:")
for i, item in enumerate(examen_dataset, 1):
    print(f"   - Processing question {i}/{len(examen_dataset)} ({item['type']})")
    item["respuesta_ft"] = model_response(my_model, my_tokenizer, item["question"], item["image_path"])

del my_model, my_tokenizer
gc.collect()
torch.cuda.empty_cache()

vram_libre = torch.cuda.mem_get_info()[0] / 1e9
print(f"VRAM Freed. Available: {vram_libre:.2f} GB")

In [ ]:
def final_evaluation_report(dataset):
    """
    Prints a detailed comparison and calculates
    the final winner based on average scores.
    """

    print("="*60)
    print("Phase 4: Evaluation report")
    print("="*60)

    # Initialize summary dictionary to track scores for both models
    summary = {'Base Model': {'acc': [], 'saf': [], 'con': []}, 'Fine-Tuned Model': {'acc': [], 'saf': [], 'con': []}}

    for i, item in enumerate(dataset, 1):
        print(f"\n{'-'*60}\nCase {i}/{len(dataset)} | Type: {item['type'].upper()}\n{'-'*60}")

        # Display the visual prompt if the question is vision-based
        if item.get("image_path"):
            display(Image(filename=item["image_path"], width=250))

        print(f"❓ Question: {item['question']}")
        print(f"🎯 Ground truth: {item['ground_truth']}")

        # Responses are anonymized as 'A' and 'B' to prevent judge bias towards a specific model name.
        order = ["A", "B"]
        random.shuffle(order)
        anon_resps = {order[0]: item['respuesta_base'], order[1]: item['respuesta_ft']}
        inv_map = {order[0]: "Base Model", order[1]: "Fine-Tuned Model"}

        # LLM Judge evaluates both responses based on the Ground Truth
        evals = evaluate_with_judge(item["question"], item["ground_truth"], anon_resps, item["image_path"])
        item['evaluations'] = {inv_map[k]: v for k, v in evals.items() if k in inv_map}

        for model_key in ['Base Model', 'Fine-Tuned Model']:
            resp = item['respuesta_base'] if model_key == 'Base Model' else item['respuesta_ft']
            scores = item['evaluations'].get(model_key, {})

            label = "🅰️ Base Model answer" if model_key == 'Base Model' else "🅱️ Fine-Tuned Model answer"
            print(f"\n{label}:\n{resp}")
            print(f"   ⭐ Scores: Acc={scores.get('accuracy',0)}, Saf={scores.get('safety',0)}, Con={scores.get('conciseness',0)}")

            # Store metrics for final average calculation
            summary[model_key]['acc'].append(scores.get('accuracy', 0))
            summary[model_key]['saf'].append(scores.get('safety', 0))
            summary[model_key]['con'].append(scores.get('conciseness', 0))

    # final section
    print(f"\n\n{'='*60}\n📊 FINAL RESULTS\n{'='*60}")
    final_totals = []
    for m in ['Base Model', 'Fine-Tuned Model']:
        avg_acc = sum(summary[m]['acc'])/len(dataset)
        avg_saf = sum(summary[m]['saf'])/len(dataset)
        avg_con = sum(summary[m]['con'])/len(dataset)
        total = (avg_acc + avg_saf + avg_con) / 3

        print(f"\n{m}:\n")
        print(f"  Medical/Visual Accuracy: {avg_acc:.2f}/10")
        print(f"  Safety:                  {avg_saf:.2f}/10")
        print(f"  Conciseness:             {avg_con:.2f}/10")
        print(f"  ━━━━━━━━━━━━━━━━━━━━━")
        print(f"  TOTAL SCORE:             {total:.2f}/10")
        final_totals.append((m, total))

    winner = max(final_totals, key=lambda x: x[1])
    print(f"\n{'='*60}\n🏆 WINNER: {winner[0]}\n    Score: {winner[1]:.2f}/10\n{'='*60}")

In [ ]:
# Execute the report
final_evaluation_report(examen_dataset)